In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ealaxi/paysim1")

print("Path to dataset files:", path)

100%|██████████| 178M/178M [00:17<00:00, 10.9MB/s] 

Extracting files...


Path to dataset files: C:\Users\GameOn\.cache\kagglehub\datasets\ealaxi\paysim1\versions\2


In [3]:
import pandas as pd

df = pd.read_csv("../data/paysim.csv")
print(df.shape)
df.head()

(6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
# How imbalanced is the data?
print(f"Fraud: {df['isFraud'].sum():,} of {len(df):,} ({df['isFraud'].mean():.4%})")

# Which transaction types have fraud?
print(df.groupby("type")["isFraud"].agg(["count", "sum", "mean"]))

# Do accounts repeat? (matters for velocity features and the graph)
print(df["nameOrig"].nunique(), df["nameDest"].nunique())
print(df["nameOrig"].value_counts().head())

Fraud: 8,213 of 6,362,620 (0.1291%)
            count   sum      mean
type                             
CASH_IN   1399284     0  0.000000
CASH_OUT  2237500  4116  0.001840
DEBIT       41432     0  0.000000
PAYMENT   2151495     0  0.000000
TRANSFER   532909  4097  0.007688
6353307 2722362
nameOrig
C1677795071    3
C1999539787    3
C724452879     3
C1976208114    3
C400299098     3
Name: count, dtype: int64


In [5]:
# Keep only the types that can contain fraud
risky = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()
print(risky.shape, risky["isFraud"].mean())

# Do the balances add up? Fraud often breaks the arithmetic.
risky["errorOrig"] = risky["oldbalanceOrg"] - risky["amount"] - risky["newbalanceOrig"]
risky["errorDest"] = risky["oldbalanceDest"] + risky["amount"] - risky["newbalanceDest"]
print(risky.groupby("isFraud")[["amount", "errorOrig", "errorDest"]].describe().T)

# Do fraudsters empty the sender's account?
risky["emptiedOrig"] = (risky["newbalanceOrig"] == 0) & (risky["oldbalanceOrg"] > 0)
print(risky.groupby("isFraud")["emptiedOrig"].mean())

# Is fraud concentrated on certain receiver accounts?
fraud = risky[risky["isFraud"] == 1]
print("Fraud txns:", len(fraud), "| unique receivers:", fraud["nameDest"].nunique())
print(fraud["nameDest"].value_counts().head())

# Fraud over time (step = 1 hour, 743 total)
print(risky.groupby("isFraud")["step"].describe())

(2770409, 11) 0.002964544224336551
isFraud                     0             1
amount    count  2.762196e+06  8.213000e+03
          mean   3.141155e+05  1.467967e+06
          std    8.771441e+05  2.404253e+06
          min    1.000000e-02  0.000000e+00
          25%    8.290823e+04  1.270913e+05
          50%    1.710345e+05  4.414234e+05
          75%    3.059942e+05  1.517771e+06
          max    9.244552e+07  1.000000e+07
errorOrig count  2.762196e+06  8.213000e+03
          mean  -2.868035e+05 -1.069233e+04
          std    8.763752e+05  2.651461e+05
          min   -9.244552e+07 -1.000000e+07
          25%   -2.804663e+05  0.000000e+00
          50%   -1.442008e+05  0.000000e+00
          75%   -5.261343e+04  0.000000e+00
          max    1.000000e-02  3.725290e-09
errorDest count  2.762196e+06  8.213000e+03
          mean  -3.091033e+04  7.325093e+05
          std    5.840943e+05  1.867748e+06
          min   -7.588573e+07 -8.875516e+06
          25%    0.000000e+00  0.000000e+

In [6]:
risky["emptiedOrig"] = (risky["newbalanceOrig"] == 0) & (risky["oldbalanceOrg"] > 0)
print(risky.groupby("isFraud")["emptiedOrig"].mean())

fraud = risky[risky["isFraud"] == 1]
print("Fraud txns:", len(fraud), "| unique receivers:", fraud["nameDest"].nunique())
print(fraud["nameDest"].value_counts().head())

isFraud
0    0.427219
1    0.975527
Name: emptiedOrig, dtype: float64
Fraud txns: 8213 | unique receivers: 8169
nameDest
C1399829166    2
C967226405     2
C1601170327    2
C650699445     2
C803116137     2
Name: count, dtype: int64
